# 02 - Silver Layer

Clean and transform the Bronze data to create trusted, high-quality tables.
- Fix data types (strings -> timestamps)
- Handle nulls and duplicates
- Standardize text
- Write to `workspace.silver` schema

In [0]:
BRONZE_SCHEMA = 'workspace.bronze'
SILVER_SCHEMA = 'workspace.silver'

print(f"Reading from: {BRONZE_SCHEMA}")
print(f"Writing to:   {SILVER_SCHEMA}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

In [0]:
from pyspark.sql.functions import col, to_timestamp
print("Cleaning orders table...")
# 1. Read from Bronze
df_orders = spark.table(f"{BRONZE_SCHEMA}.orders")

##Converting date from string to timestamp

In [0]:
# 1. Read from Bronze
df_orders = spark.table(f"{BRONZE_SCHEMA}.orders")

# 2. Transform: Cast string columns to proper timestamps
df_orders_clean = df_orders \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .dropDuplicates(["order_id"]) # Remove any duplicate orders

# 3. Write to Silver
df_orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_SCHEMA}.orders")
print("✅ Silver 'orders' table created!")
display(spark.table(f"{SILVER_SCHEMA}.orders").limit(5))


## cleaning customer table

In [0]:
from pyspark.sql.functions import col, trim, lower, initcap

print("Cleaning customers table...")

df_customers = spark.table(f"{BRONZE_SCHEMA}.customers")

df_customers_clean = df_customers \
    .withColumn("customer_city", initcap(trim(lower(col("customer_city"))))) \
    .withColumn("customer_state", trim(col("customer_state"))) \
    .dropDuplicates(["customer_id"])

df_customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_SCHEMA}.customers")

print("✅ Silver 'customers' table created!")
display(spark.table(f"{SILVER_SCHEMA}.customers").limit(5))

In [0]:
print("Cleaning and enriching products table...")

# 1. Load both tables from Bronze
df_products = spark.table(f"{BRONZE_SCHEMA}.products")

# Load translation table, but only keep the two columns we actually need!
df_translation = spark.table(f"{BRONZE_SCHEMA}.product_category_translation") \
    .select("product_category_name", "product_category_name_english")

# 2. Join them to get the English category name
df_products_clean = df_products.join(
    df_translation,
    on="product_category_name",
    how="left"
).dropDuplicates(["product_id"])

# 3. Write to Silver
df_products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{SILVER_SCHEMA}.products")

print("✅ Silver 'products' table created (with English categories)!")
display(spark.table(f"{SILVER_SCHEMA}.products").select("product_id", "product_category_name", "product_category_name_english").limit(5))

##Pointing the Portuguese tables to the English table for tanslation

In [0]:
print("Cleaning remaining tables...")

remaining_tables = [
    "order_items",
    "order_payments",
    "order_reviews",
    "sellers",
    "geolocation"
]

for table_name in remaining_tables:
    print(f"⏳ Processing {table_name}...")
    
    # 1. Read from Bronze
    df = spark.table(f"{BRONZE_SCHEMA}.{table_name}")
    
    # 2. Clean: Drop exact duplicate rows
    df_clean = df.dropDuplicates()
    
    # 3. Write to Silver
    df_clean.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{SILVER_SCHEMA}.{table_name}")
    
    print(f"  ✅ {SILVER_SCHEMA}.{table_name} saved!")

print("\n🎉 All Silver tables created successfully!")